# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os.path as osp
import pandas as pd
import json
import joblib

from ml_fmda.moisture_rnn import OperationalRNNPredictor, TimeWarpedFuelClassPredictors, predict_auto_batch
from ml_fmda.data import scale_3d
from ml_fmda.utils import read_yml, Dict

In [ ]:
# Path to pretrained model objects, unzipped from fm_transfer.zip
MODEL_DIR = "/Users/hirschij/Documents/Projects/Wildfire/ml_fmda/models/fm_transfer"
# Path to test input gridded data
DATA_DIR = "/Users/hirschij/Documents/Projects/Wildfire/ml_fmda/models/ml_fmda_test_data"

In [ ]:
# Model params
params = Dict(read_yml(osp.join(MODEL_DIR, "fm10_source_model", "params.yaml")))
print(f"{params.features_list=}")
print(f"{params.hidden_layers=}")
print(f"{params.hidden_units=}")
print(f"{params.output_dimension=}")

## Build FM10 model from Pretrained Weights

Chosen as seed with median accuracy, so representative of the replications. The `OperationalRNNPredictor` class can build a model from a params file and weights path. 

Model constructor will ignore parameters related to training, such as timesteps, batch size, learning rate, etc. During active prediction, model is deployed pointwise on arbitrary number of locations and arbitrary timesteps

In [ ]:
median_seed = pd.read_csv(osp.join(MODEL_DIR, "fm10_source_model", "median_seed.csv")).seed[0]
print(f"median seed: {median_seed}")

In [ ]:
rnn = OperationalRNNPredictor.from_weights(
    params=params,
    weights_path=osp.join(MODEL_DIR, "fm10_source_model", f"seed_{median_seed}", "rnn.weights.h5")
)

In [ ]:
rnn.summary()

## Build other fuel class models from Time-Warps

From the FM10 model weights (specific to a particular seed), find the time-warp params associated with those weights and use to build predictors for other fuel classes. The class `TimeWarpedFuelClassPredictors` can build models for FM1, FM10, FM100, and FM1000 from a params object, weights path (for FM10 weights), and structured time warp params for each fuel class.

The class creates an OperationalRNNPredictor for each fuel class. The FM10 is constructed the same as previous, and the weights for the other fuel classes are obtained through the time-warping method, which shifts the input and forget gate biases.

The time-warps can be reported with the associated RMSE when evaluated on a test set of the OK field study data. These RMSE should be representative of the 100 seeds.

In [ ]:
# Get time-warp params for target replication seed
twarp_summary = pd.read_csv(osp.join(MODEL_DIR, "fm_target_models", "all_seeds_summary.csv"))
twarp_summary[twarp_summary.seed == median_seed].transpose()

In [ ]:
row = twarp_summary[twarp_summary.seed == median_seed].iloc[0]

warps = {
    "fm1": (row["bi_1"], row["bf_1"]),
    "fm100": (row["bi_100"], row["bf_100"]),
    "fm1000": (row["bi_1000"], row["bf_1000"]),
}

In [ ]:
transfer_predictor = TimeWarpedFuelClassPredictors(
    params=params,
    weights_path=osp.join(MODEL_DIR, "fm10_source_model", f"seed_{median_seed}", "rnn.weights.h5"),
    warps=warps
)
transfer_predictor.predictors

## Check with Simulated Data

`ml_fmda_test_data.zip` contains a test gridded dataset, derived from HRRR inputs, and an associated features list should match current model features list.

Shape of `X_gridded.npy` is `(ny*nw, ntime, nfeatures)`

Steps:
* Apply model scaler
* Run prediction
* Reshape to spatial grid

For prediction, we use a utility function `predict_auto_batch`. During active prediction, the batch size is purely an optimization parameter. The larger means more unique locations are run simultaneously, but can cause memory issues. The `predict_auto_batch` function tries large batch sizes and then steps down if memory issues are hit. This should not affect resulting predictions (see package test function `test_cycle_batch_size`)

In [ ]:
with open(osp.join(DATA_DIR, "features.json")) as f:
    features_list = json.load(f)

assert features_list == params.features_list

X_gridded = np.load(osp.join(DATA_DIR, "X_gridded.npy"))
lons = np.load(osp.join(DATA_DIR, "lon.npy"))
lats = np.load(osp.join(DATA_DIR, "lat.npy"))
print(f"{X_gridded.shape=}")

In [ ]:
# Get scaler and apply
scaler = joblib.load(osp.join(MODEL_DIR, "fm10_source_model", f"seed_{median_seed}", "scaler.joblib"))
scaler.mean_

In [ ]:
# Use utility to apply fitted scaler to a 3d object
X = scale_3d(X_gridded, scaler)

In [ ]:
# Predictions

fm10 = predict_auto_batch(rnn, X)
print(f"{fm10.shape=}")

In [ ]:
# All class predictions
preds = transfer_predictor.predict(X)
print(f"{preds.shape=}")

In [ ]:
# FM10 predictions should match to machine error
assert np.allclose(fm10.squeeze(), preds[:,:,1].squeeze())

In [ ]:
# All class cycle predictions
preds_cycle = transfer_predictor.predict_cycle(X)
print(f"{preds_cycle.shape=}")

In [ ]:
# FM10 predictions should match to machine error
assert np.allclose(preds_cycle, preds)

## Viz

In [ ]:
assert X_gridded.ndim == 3
assert lats.shape == lons.shape
assert X_gridded.shape[0] == lats.size

ny, nx = lats.shape
ntime = X_gridded.shape[1]
nfeatures = X_gridded.shape[2]

assert fm10.shape[:2] == X_gridded.shape[:2]
assert fm10.shape[2] == 1

fm10_grid = fm10.reshape(ny, nx, ntime, 1)

X_grid = X_gridded.reshape(ny, nx, ntime, nfeatures)
print(f"{X_grid.shape=}")
print(f"{fm10_grid.shape=}")

In [ ]:
preds_cycle_grid = preds_cycle.reshape(ny, nx, ntime, 4)
print(f"{preds_cycle_grid.shape=}")

In [ ]:
plt.plot(preds_cycle_grid[50,100,4:,0], label="FM1")
plt.plot(preds_cycle_grid[50,100,4:,1], label="FM10")
plt.plot(preds_cycle_grid[50,100,4:,2], label="FM100")
plt.plot(preds_cycle_grid[50,100,4:,3], label="FM1000")
plt.title("Fuel Moisture Predictions at Grid Cell (y=50, x=100)")
plt.xlabel("Timestep")
plt.ylabel("Fuel Moisture")
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(7, 16))

fuel_classes = ["FM1", "FM10", "FM100", "FM1000"]

for i, fuel_class in enumerate(fuel_classes):
    im = axes[i].imshow(
        preds_cycle_grid[:, :, 20, i],
        origin="lower",
        cmap="RdBu_r",
        extent=[lons.min(), lons.max(), lats.min(), lats.max()],
        aspect="auto",
    )
    axes[i].set_title(fuel_class)
    axes[i].set_xlabel("Longitude")
    axes[i].set_ylabel("Latitude")
    fig.colorbar(im, ax=axes[i], label=fuel_class)

plt.tight_layout()
plt.show()

In [ ]:
# from matplotlib.colors import Normalize
# from matplotlib.cm import ScalarMappable

# norm = Normalize(vmin=0, vmax=35)

# fig, ax = plt.subplots()

# im = ax.imshow(
#     fm10_grid[:, :, 0, 0],
#     origin="lower",
#     cmap="RdBu_r",
#     norm=norm,
#     extent=[lons.min(), lons.max(), lats.min(), lats.max()],
#     aspect="auto",
#     interpolation="nearest",
# )

# # Fixed colorbar, independent of the animation
# sm = ScalarMappable(norm=norm, cmap="RdBu_r")
# fig.colorbar(sm, ax=ax, label="FM10")

# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")

# def update(t):
#     im.set_data(fm10_grid[:, :, t, 0])
#     ax.set_title(f"Time step {t}")
#     return (im,)

# anim = FuncAnimation(
#     fig,
#     update,
#     frames=fm10_grid.shape[2],
#     interval=150,
# )

# anim.save("fm10.gif", writer=PillowWriter(fps=5))
# plt.close(fig)